# V18 BCR/TCR Column Exploration
**Purpose:** Identify all BCR/TCR-related columns in h5ad, their data types, coverage, and structure.
**Date:** 2026-03-14
**Priority:** Understand what data is available before designing IT-oriented analysis.

In [1]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.9 MB/s eta 0:00:00
  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total


In [2]:
# Cell 1: Mount Drive & Load h5ad (backed mode)
from google.colab import drive
drive.mount('/content/drive')

import scanpy as sc
import pandas as pd
import numpy as np

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
adata = sc.read_h5ad(DATA_PATH, backed='r')
print(f'Total cells: {adata.n_obs:,}')
print(f'Total genes: {adata.n_vars:,}')
print(f'\nAll .obs columns ({len(adata.obs.columns)}):')
for i, col in enumerate(adata.obs.columns):
    print(f'  {i:3d}. {col} — dtype: {adata.obs[col].dtype}')

Mounted at /content/drive
Total cells: 243,000
Total genes: 24,452

All .obs columns (34):
    0. sample — dtype: category
    1. tissue — dtype: category
    2. Stage — dtype: category
    3. IT_cluster_21 — dtype: float64
    4. IT_cluster_23 — dtype: float64
    5. IT_cluster_25 — dtype: float64
    6. IT_nk_collapse — dtype: float32
    7. IT_IT_signature — dtype: float64
    8. GSM_ID — dtype: category
    9. IT_score_v2 — dtype: float64
   10. IT_score_v3 — dtype: float64
   11. IT_score_v4 — dtype: float64
   12. IT_signature_final — dtype: float64
   13. IT_like — dtype: category
   14. PW_mTOR_signaling — dtype: float32
   15. PW_glycolysis — dtype: float32
   16. PW_oxidative_phosphorylation — dtype: float32
   17. PW_nk_cell_cytotoxicity — dtype: float32
   18. PW_il15_signaling — dtype: float32
   19. PW_b_cell_differentiation — dtype: float32
   20. leiden — dtype: int64
   21. gut2021_subcluster — dtype: category
   22. major_lineage — dtype: category
   23. gut2021_subcl

In [3]:
# Cell 2: Find BCR/TCR/VDJ/Ig-related columns
import re

keywords = ['bcr', 'tcr', 'vdj', 'clone', 'clono', 'chain', 'cdr3',
            'igh', 'igk', 'igl', 'tra_', 'trb_', 'trd_', 'trg_',
            'contig', 'productive', 'barcode', 'ir_', 'receptor',
            'isotype', 'c_gene', 'v_gene', 'j_gene', 'd_gene',
            'immune_receptor', 'has_ir', 'multi_chain']

vdj_cols = []
for col in adata.obs.columns:
    col_lower = col.lower()
    for kw in keywords:
        if kw in col_lower:
            vdj_cols.append(col)
            break

print(f'=== BCR/TCR/VDJ-related columns found: {len(vdj_cols)} ===')
for col in vdj_cols:
    series = adata.obs[col]
    n_notna = series.notna().sum()
    n_total = len(series)
    pct = n_notna / n_total * 100
    if series.dtype == 'category' or series.dtype == 'object':
        nunique = series.dropna().nunique()
        top3 = series.value_counts().head(3).to_dict()
        print(f'\n  {col}')
        print(f'    dtype={series.dtype}, non-NA={n_notna:,} ({pct:.1f}%), unique={nunique}')
        print(f'    top3: {top3}')
    else:
        print(f'\n  {col}')
        print(f'    dtype={series.dtype}, non-NA={n_notna:,} ({pct:.1f}%)')
        if n_notna > 0:
            print(f'    range: [{series.min()}, {series.max()}], mean={series.mean():.4f}')

if len(vdj_cols) == 0:
    print('\n⚠️ No BCR/TCR columns found by keyword search.')
    print('Listing ALL columns for manual inspection:')
    for col in adata.obs.columns:
        print(f'  {col}')

=== BCR/TCR/VDJ-related columns found: 10 ===

  TCR_clone.id
    dtype=category, non-NA=64,284 (26.5%), unique=40517
    top3: {'clone_D529074:3986:771': 746, 'clone_P190716:11587:684': 670, 'clone_D529074:4617:412': 396}

  TCR_v_gene.x
    dtype=category, non-NA=64,284 (26.5%), unique=44
    top3: {'TRAV1-2': 7063, 'TRAV12-2': 3555, 'TRAV13-1': 3287}

  TCR_j_gene.x
    dtype=category, non-NA=64,284 (26.5%), unique=53
    top3: {'TRAJ33': 5991, 'TRAJ43': 2342, 'TRAJ45': 2267}

  TCR_cdr3_nt.x
    dtype=category, non-NA=64,284 (26.5%), unique=36655
    top3: {'TGTGCAGCCCCAATGGAATATGGAAACAAACTGGTCTTT': 746, 'TGTGCTGTGAGAGATAGCAACTATCAGTTAATCTGG': 699, 'TGTGCCGTGAACATAGATACAGGAGGAGGTGCTGACGGACTCACCTTT': 671}

  TCR_CType
    dtype=category, non-NA=64,243 (26.4%), unique=1
    top3: {'TRAC': 64243}

  BCR_clone.id
    dtype=category, non-NA=12,764 (5.3%), unique=12461
    top3: {'clone_P190911:12276:102': 85, 'clone_P190911:12099:20': 18, 'clone_D529074:2179:11': 10}

  BCR_v_gene
    d

In [4]:
# Cell 3: Check .var for Ig/TR gene names (BCR/TCR genes in expression matrix)
ig_genes = [g for g in adata.var_names if re.match(r'^IG[HKL][VDJCGMADEK]', g)]
tr_genes = [g for g in adata.var_names if re.match(r'^TR[ABDG][VDJC]', g)]

print(f'=== Immunoglobulin (Ig) genes in var_names: {len(ig_genes)} ===')
if ig_genes:
    # Group by chain type
    igh = [g for g in ig_genes if g.startswith('IGH')]
    igk = [g for g in ig_genes if g.startswith('IGK')]
    igl = [g for g in ig_genes if g.startswith('IGL')]
    print(f'  IGH (heavy): {len(igh)} — {igh[:10]}...' if len(igh)>10 else f'  IGH (heavy): {len(igh)} — {igh}')
    print(f'  IGK (kappa): {len(igk)} — {igk[:10]}...' if len(igk)>10 else f'  IGK (kappa): {len(igk)} — {igk}')
    print(f'  IGL (lambda): {len(igl)} — {igl[:10]}...' if len(igl)>10 else f'  IGL (lambda): {len(igl)} — {igl}')

print(f'\n=== TCR genes in var_names: {len(tr_genes)} ===')
if tr_genes:
    tra = [g for g in tr_genes if g.startswith('TRA')]
    trb = [g for g in tr_genes if g.startswith('TRB')]
    trd = [g for g in tr_genes if g.startswith('TRD')]
    trg = [g for g in tr_genes if g.startswith('TRG')]
    print(f'  TRA (alpha): {len(tra)}')
    print(f'  TRB (beta): {len(trb)}')
    print(f'  TRD (delta): {len(trd)}')
    print(f'  TRG (gamma): {len(trg)}')

# Also check constant region genes specifically
const_genes = ['IGHM', 'IGHD', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4',
               'IGHA1', 'IGHA2', 'IGHE', 'IGKC', 'IGLC1', 'IGLC2', 'IGLC3',
               'JCHAIN', 'MZB1', 'SDC1', 'CD38', 'XBP1', 'PRDM1']
print(f'\n=== Key B cell / Plasma cell marker genes presence ===')
for g in const_genes:
    present = g in adata.var_names
    print(f'  {g}: {"✅" if present else "❌"}')

=== Immunoglobulin (Ig) genes in var_names: 227 ===
  IGH (heavy): 104 — ['IGHEP2', 'IGHMBP2', 'IGHA2', 'IGHE', 'IGHG4', 'IGHG2', 'IGHGP', 'IGHA1', 'IGHG1', 'IGHG3']...
  IGK (kappa): 66 — ['IGKV1OR1-1', 'IGKV3OR2-268', 'IGKC', 'IGKJ5', 'IGKJ1', 'IGKV4-1', 'IGKV5-2', 'IGKV7-3', 'IGKV2-4', 'IGKV1-5']...
  IGL (lambda): 57 — ['IGLVI-70', 'IGLV4-69', 'IGLV10-54', 'IGLV8-61', 'IGLV4-60', 'IGLV6-57', 'IGLV11-55', 'IGLV5-52', 'IGLV1-51', 'IGLV1-50']...

=== TCR genes in var_names: 199 ===
  TRA (alpha): 100
  TRB (beta): 72
  TRD (delta): 7
  TRG (gamma): 20

=== Key B cell / Plasma cell marker genes presence ===
  IGHM: ✅
  IGHD: ✅
  IGHG1: ✅
  IGHG2: ✅
  IGHG3: ✅
  IGHG4: ✅
  IGHA1: ✅
  IGHA2: ✅
  IGHE: ✅
  IGKC: ✅
  IGLC1: ❌
  IGLC2: ✅
  IGLC3: ✅
  JCHAIN: ✅
  MZB1: ✅
  SDC1: ✅
  CD38: ✅
  XBP1: ✅
  PRDM1: ✅


In [5]:
# Cell 4: Check .uns and .obsm for any VDJ-related stored data
print('=== .uns keys ===')
for key in sorted(adata.uns.keys()):
    val = adata.uns[key]
    val_type = type(val).__name__
    if isinstance(val, (dict, pd.DataFrame)):
        size = len(val)
    elif isinstance(val, np.ndarray):
        size = val.shape
    else:
        size = ''
    kw_match = any(kw in key.lower() for kw in ['bcr','tcr','vdj','clone','ir_','receptor'])
    flag = ' ⭐ VDJ-RELATED' if kw_match else ''
    print(f'  {key} ({val_type}, {size}){flag}')

print(f'\n=== .obsm keys ===')
for key in sorted(adata.obsm.keys()):
    val = adata.obsm[key]
    print(f'  {key} — shape: {val.shape}')

print(f'\n=== .obsp keys ===')
if hasattr(adata, 'obsp') and adata.obsp:
    for key in sorted(adata.obsp.keys()):
        print(f'  {key}')
else:
    print('  (none)')

=== .uns keys ===
  Stage_colors (ndarray, (5,))
  leiden_temp (dict, 2)
  neighbors (dict, 3)
  pca (dict, 3)
  umap (dict, 1)

=== .obsm keys ===
  X_pca — shape: (243000, 50)
  X_umap — shape: (243000, 2)

=== .obsp keys ===
  connectivities
  distances


In [6]:
# Cell 5: If VDJ columns exist, cross-tabulate with Stage × Tissue × Lineage
# This gives us the IT-oriented view immediately

obs = adata.obs.copy()

# Derive tissue and donor columns
if 'tissue' in obs.columns:
    tissue_col = 'tissue'
elif 'Tissue' in obs.columns:
    tissue_col = 'Tissue'
else:
    for c in obs.columns:
        if 'tissue' in c.lower():
            tissue_col = c
            break

stage_col = 'Stage'  # known from prior analysis

# Attempt to find BCR/TCR presence columns
# Common patterns: 'has_ir', 'receptor_type', 'chain_pairing', etc.
has_bcr_col = None
has_tcr_col = None
for col in vdj_cols:
    cl = col.lower()
    if 'has' in cl and 'ir' in cl:
        print(f'Found has_ir column: {col}')
        print(obs[col].value_counts())
    if 'receptor_type' in cl or 'chain_type' in cl:
        print(f'\nFound receptor/chain type column: {col}')
        print(obs[col].value_counts())
    if 'clone' in cl:
        print(f'\nFound clone column: {col}')
        n_notna = obs[col].notna().sum()
        n_unique = obs[col].dropna().nunique()
        print(f'  non-NA: {n_notna:,}, unique: {n_unique:,}')
        # Cross-tab with Stage
        if n_notna > 0:
            clone_by_stage = obs.groupby(stage_col)[col].apply(
                lambda x: pd.Series({
                    'n_with_clone': x.notna().sum(),
                    'n_unique_clones': x.dropna().nunique(),
                    'pct_with_clone': x.notna().mean() * 100
                })
            ).unstack()
            print(clone_by_stage)

# If no VDJ columns found, check if the Feb 5 analysis used external files
if len(vdj_cols) == 0:
    print('\n⚠️ No VDJ columns in .obs.')
    print('BCR/TCR data may be in separate files (filtered_contig_annotations.csv).')
    print('Checking Drive for these files...')
    import os
    base = '/content/drive/MyDrive/ITLAS/data'
    for root, dirs, files in os.walk(base):
        for f in files:
            fl = f.lower()
            if any(kw in fl for kw in ['contig', 'vdj', 'bcr', 'tcr', 'clonotype']):
                fpath = os.path.join(root, f)
                fsize = os.path.getsize(fpath)
                print(f'  Found: {fpath} ({fsize/1024:.1f} KB)')


Found clone column: TCR_clone.id
  non-NA: 64,284, unique: 40,517
       n_with_clone  n_unique_clones  pct_with_clone
Stage                                               
CR              0.0              0.0        0.000000
AR           4614.0           2840.0       10.151368
IA          29957.0          19349.0       47.896714
IT          18808.0          13158.0       38.243966
NL          10905.0           5170.0       25.611217

Found clone column: BCR_clone.id
  non-NA: 12,764, unique: 12,461
       n_with_clone  n_unique_clones  pct_with_clone
Stage                                               
CR              0.0              0.0        0.000000
AR            606.0            605.0        1.333275
IA           5192.0           5010.0        8.301223
IT           3985.0           3953.0        8.103052
NL           2981.0           2893.0        7.001104


/tmp/ipykernel_3628/2619234829.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  clone_by_stage = obs.groupby(stage_col)[col].apply(
/tmp/ipykernel_3628/2619234829.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  clone_by_stage = obs.groupby(stage_col)[col].apply(


In [7]:
# Cell 6: Also scan the broader ITLAS folder for any BCR/TCR result files
import os

search_roots = [
    '/content/drive/MyDrive/ITLAS/data',
    '/content/drive/MyDrive/ITLAS/results',
]

vdj_keywords = ['contig', 'vdj', 'bcr', 'tcr', 'clonotype', 'clonal',
                'repertoire', 'immunoglobulin', 'receptor']

print('=== Scanning ITLAS folders for BCR/TCR files ===')
found_files = []
for base in search_roots:
    if not os.path.exists(base):
        print(f'  Path not found: {base}')
        continue
    for root, dirs, files in os.walk(base):
        for f in files:
            fl = f.lower()
            if any(kw in fl for kw in vdj_keywords):
                fpath = os.path.join(root, f)
                fsize = os.path.getsize(fpath)
                found_files.append((fpath, fsize))
                print(f'  {fpath} ({fsize/1024:.1f} KB)')

if not found_files:
    print('  No BCR/TCR files found in data/ or results/ folders.')
    print('\n  → Files may be in raw/ subfolder or need to be downloaded from GEO.')
    # Check raw folder structure
    raw_path = '/content/drive/MyDrive/ITLAS/data/raw'
    if os.path.exists(raw_path):
        print(f'\n  raw/ folder contents:')
        for item in os.listdir(raw_path):
            full = os.path.join(raw_path, item)
            if os.path.isdir(full):
                n_files = len(os.listdir(full))
                print(f'    📁 {item}/ ({n_files} files)')
            else:
                print(f'    📄 {item} ({os.path.getsize(full)/1024:.1f} KB)')
    else:
        print(f'  raw/ folder does not exist at {raw_path}')

print(f'\nTotal BCR/TCR related files found: {len(found_files)}')

=== Scanning ITLAS folders for BCR/TCR files ===
  No BCR/TCR files found in data/ or results/ folders.

  → Files may be in raw/ subfolder or need to be downloaded from GEO.

  raw/ folder contents:
    📄 GSE182159_batch_0.h5ad (2986527.4 KB)
    📄 GSE182159_batch_1.h5ad (3536641.0 KB)
    📄 GSE182159_batch_2.h5ad (4295049.4 KB)
    📄 GSE182159_batch_3.h5ad (7728080.4 KB)
    📄 GSE182159_batch_4.h5ad (4686768.5 KB)

Total BCR/TCR related files found: 0


In [8]:
# Cell 7: Quick summary — what do we have and what do we need?
print('=' * 70)
print('SUMMARY: BCR/TCR Data Availability in GSE182159 h5ad')
print('=' * 70)
print(f'\n1. VDJ-related .obs columns: {len(vdj_cols)}')
if vdj_cols:
    for c in vdj_cols:
        print(f'   - {c}')
print(f'\n2. Ig genes in expression matrix: {len(ig_genes)}')
print(f'   TCR genes in expression matrix: {len(tr_genes)}')
print(f'\n3. External VDJ files found: {len(found_files)}')

print('\n' + '=' * 70)
print('NEXT STEPS for IT-Oriented BCR/TCR Analysis:')
print('=' * 70)
print('''
If VDJ columns exist in .obs:
  → Tissue-separated (Liver vs Blood) clonality by Stage
  → Donor-level BCR/TCR metrics → Mann-Whitney IT vs NL
  → Isotype distribution (IgM/IgG/IgA) by Stage × Tissue
  → Link to B/PlasmaB subcluster annotations

If VDJ columns NOT in .obs but external files exist:
  → Load filtered_contig_annotations.csv
  → Match cell barcodes to h5ad
  → Add VDJ metadata to .obs
  → Then proceed with above analyses

If NO VDJ data available locally:
  → Download from GEO GSE182159 supplementary
  → Or use TRUST4 to reconstruct from GEX BAM files
''')

SUMMARY: BCR/TCR Data Availability in GSE182159 h5ad

1. VDJ-related .obs columns: 10
   - TCR_clone.id
   - TCR_v_gene.x
   - TCR_j_gene.x
   - TCR_cdr3_nt.x
   - TCR_CType
   - BCR_clone.id
   - BCR_v_gene
   - BCR_j_gene
   - BCR_cdr3_nt
   - BCR_CType

2. Ig genes in expression matrix: 227
   TCR genes in expression matrix: 199

3. External VDJ files found: 0

NEXT STEPS for IT-Oriented BCR/TCR Analysis:

If VDJ columns exist in .obs:
  → Tissue-separated (Liver vs Blood) clonality by Stage
  → Donor-level BCR/TCR metrics → Mann-Whitney IT vs NL
  → Isotype distribution (IgM/IgG/IgA) by Stage × Tissue
  → Link to B/PlasmaB subcluster annotations

If VDJ columns NOT in .obs but external files exist:
  → Load filtered_contig_annotations.csv
  → Match cell barcodes to h5ad
  → Add VDJ metadata to .obs
  → Then proceed with above analyses

If NO VDJ data available locally:
  → Download from GEO GSE182159 supplementary
  → Or use TRUST4 to reconstruct from GEX BAM files

